# SentimentIQ — LSTM-Based Intelligent Sentiment Analysis System
### End-to-End Deep Learning Pipeline for 3-Class Customer Review Analytics

This notebook demonstrates the complete end-to-end lifecycle:
1. **Dataset Ingestion & Exploratory Data Analysis (EDA)**
2. **Text Preprocessing & Negation-Aware Normalization**
3. **Zero-Leakage Stratified Splitting**
4. **Training-Set Vocabulary Tokenization & Sequence Padding**
5. **LSTM Deep Learning Architecture Construction**
6. **Model Training with EarlyStopping & Checkpointing**
7. **Baseline Model (TF-IDF + Logistic Regression)**
8. **Evaluation on Untouched Test Set & Confusion Matrix**
9. **Error Diagnostics & Qualitative Analysis**
10. **Inference Pipeline & Multi-Aspect Sentiment Decomposition**

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
sys.path.append(os.path.abspath('..'))

from src.config import (
    RAW_DATASET_PATH,
    MAX_WORDS, MAX_LENGTH, EMBEDDING_DIM, LSTM_UNITS,
    DROPOUT_RATE, DENSE_UNITS, LEARNING_RATE, BATCH_SIZE, EPOCHS,
    LABEL_TO_SENTIMENT, CLASS_NAMES
)
from src.preprocessing import clean_text
from src.data_loader import load_or_generate_dataset, perform_eda, split_and_save_data
from src.model import ReviewTokenizer, build_lstm_model
from src.predict import SentimentPredictor
from src.aspect_analyzer import extract_aspect_sentiments

## 1. Load Dataset & Perform Exploratory Data Analysis (EDA)

In [ ]:
df = load_or_generate_dataset()
print(f"Total raw dataset shape: {df.shape}")
df.head()

In [ ]:
eda_summary = perform_eda(df)
print(json.dumps(eda_summary, indent=2))

## 2. Text Preprocessing & Negation Preservation
Verify that sentiment-reversing words (e.g. `not`, `never`, `no`) are carefully preserved.

In [ ]:
sample_raw = "<p>I didn't like the delayed shipping, but it's <b>not bad</b> at all! http://review.link</p>"
sample_cleaned = clean_text(sample_raw)
print(f"Raw:     {sample_raw}")
print(f"Cleaned: {sample_cleaned}")

## 3. Stratified Dataset Splitting (80% Train, 10% Val, 10% Test)

In [ ]:
train_df, val_df, test_df = split_and_save_data(df)
print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")

## 4. Tokenization and Sequence Padding (Fitted on Train Set ONLY)

In [ ]:
tokenizer = ReviewTokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["cleaned_review"].tolist())

train_seqs = tokenizer.texts_to_sequences(train_df["cleaned_review"].tolist())
val_seqs = tokenizer.texts_to_sequences(val_df["cleaned_review"].tolist())
test_seqs = tokenizer.texts_to_sequences(test_df["cleaned_review"].tolist())

X_train = tokenizer.pad_sequences(train_seqs, maxlen=MAX_LENGTH)
X_val = tokenizer.pad_sequences(val_seqs, maxlen=MAX_LENGTH)
X_test = tokenizer.pad_sequences(test_seqs, maxlen=MAX_LENGTH)

y_train = np.array(train_df["sentiment"], dtype=np.int32)
y_val = np.array(val_df["sentiment"], dtype=np.int32)
y_test = np.array(test_df["sentiment"], dtype=np.int32)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

## 5. Build and Train LSTM Deep Learning Model

In [ ]:
vocab_size = min(len(tokenizer.word_index) + 2, MAX_WORDS)
model = build_lstm_model(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    max_length=MAX_LENGTH,
    lstm_units=LSTM_UNITS,
    dropout_rate=DROPOUT_RATE,
    dense_units=DENSE_UNITS,
    num_classes=3,
    learning_rate=LEARNING_RATE
)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)

## 6. Evaluate on Untouched Test Set & Compare with Baseline

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_probs = model.predict(X_test)
y_preds = np.argmax(y_probs, axis=1)

print(f"Test Accuracy: {accuracy_score(y_test, y_preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_preds, target_names=CLASS_NAMES))

## 7. Aspect-Based Sentiment Analysis Extension

In [ ]:
test_mixed_review = "The camera quality is excellent but the battery life is terrible."
aspect_results = extract_aspect_sentiments(test_mixed_review)
print(f"Review: {test_mixed_review}\n")
for asp in aspect_results:
    print(f"● Aspect: {asp['aspect']} -> Sentiment: {asp['sentiment']} (Clause: '{asp['clause']}')")